# Notebook 3 — Pipeline Active Learning (Roth & Small 2008)

This notebook demonstrates the **multi-stage active learning algorithm** described in:
> *Active Learning for Pipeline Models*, D. Roth & K. Small, AAAI 2008.

## Why pipeline AL?

The inspection system is not a single model — it is a **sequential pipeline**:

```
Raw image  →  Stage 1: Detection   (is there a defect?)  →  Stage 2: Classification  (which type?)
```

Errors in Stage 1 cascade into Stage 2. Giving annotation budget equally to both
stages wastes it if Stage 1 is still unreliable. The Roth & Small algorithm solves
this by maintaining **adaptive per-stage weights β** that shift the budget:

- **Early cycles:** β = [1, 0] — all budget to detection
- **Later cycles:** β shifts toward [0, 1] — budget moves to classification once detection stabilises

## What this notebook shows

1. Load the 480 synthetic steel coil images
2. Build a two-stage pipeline (detection + classification)
3. Simulate 8 active learning cycles
4. Visualise how β shifts over time
5. Compare per-stage uncertainty distributions
6. Show combined weighted scores vs. individual stage scores

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

# Make sure the repository root is on sys.path
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from steeldefect_active_learning.data.dataset import SteelDefectDataset
from steeldefect_active_learning.models.model_loader import load_model
from steeldefect_active_learning.active_learning.pipeline_al import PipelineActiveLearner

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Load the Steel Coil Image Dataset

In [ ]:
DATA_DIR = ROOT / "data" / "sample_images"

if not DATA_DIR.exists():
    print("Generating sample data...")
    os.system(f'python "{ROOT / "data" / "generate_sample_data.py"}" --n_per_class 80 --size 128')

dataset = SteelDefectDataset(
    source=str(DATA_DIR),
    image_size=(128, 128),
    extensions=[".png"],
)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Dataset: {len(dataset)} images")
print(f"Batches: {len(loader)}")

# Show 6 sample images (one per class)
CLASS_NAMES = ["scratch", "pit", "roll_mark", "edge_crack", "inclusion", "clean"]
fig, axes = plt.subplots(1, 6, figsize=(15, 2.5))
for ax, cls in zip(axes, CLASS_NAMES):
    samples = list((DATA_DIR / cls).glob("*.png"))
    if samples:
        from PIL import Image
        img = np.array(Image.open(samples[0]).convert("L"))
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(cls, fontsize=9)
    ax.axis("off")
plt.suptitle("Steel Coil Surface — One Sample per Defect Class", fontsize=11)
plt.tight_layout()
plt.show()

## 2. Build the Two-Stage Pipeline

| Stage | Task | Classes | Notes |
|-------|------|---------|-------|
| 0 — Detection | Binary classification | 2 (clean / defective) | Must be reliable before Stage 1 is useful |
| 1 — Classification | Multi-class | 6 (scratch, pit, ...) | Depends on Stage 0 predictions |

We use random-weight ResNet18 models here (no real training data) to demonstrate
the algorithm mechanics. In production, replace `checkpoint=None` with trained weights.

In [ ]:
model_detection     = load_model("resnet18", checkpoint=None, num_classes=2, device=device)
model_classification = load_model("resnet18", checkpoint=None, num_classes=6, device=device)

print("Stage 0 — Detection     : ResNet18  (2 classes: clean / defective)")
print("Stage 1 — Classification: ResNet18  (6 classes: scratch/pit/roll_mark/edge_crack/inclusion/clean)")
print("(random weights — demo only)")

## 3. Initialise PipelineActiveLearner

The learner starts with **β = [1, 0]** — all annotation budget flows to Stage 0 (detection).
After each cycle it computes ΔU (change in average query score) and updates β proportionally.

In [ ]:
learner = PipelineActiveLearner(
    models=[model_detection, model_classification],
    tasks=["classification", "classification"],
    uncertainty_methods=["entropy", "entropy"],
    update_rate=0.15,   # lambda — higher = faster beta shift
)

print(learner.summary())

## 4. Simulate Active Learning Cycles

Each cycle represents one round of:
1. Select candidates using current β-weighted score
2. **(In production)** Send to annotators, retrain models
3. Update β based on how much uncertainty changed in each stage

Here we skip retraining (random weights stay fixed), so all uncertainty changes
are noise-driven — but the β update mechanics are fully demonstrated.

In [ ]:
N_CYCLES = 8
BUDGET_PER_CYCLE = 30

cycle_records = []

for cycle in range(N_CYCLES):
    candidates = learner.select_candidates(loader, budget=BUDGET_PER_CYCLE, device=device)
    cycle_records.append({
        "cycle": cycle + 1,
        "n_selected": len(candidates),
        "beta_detection": float(learner.beta[0]),
        "beta_classification": float(learner.beta[1]),
    })
    # beta update: step() would normally run AFTER retraining
    learner.step(dataloader=loader, device=device)
    print(f"Cycle {cycle+1:2d}  |  beta={learner.beta.round(3)}  |  selected={len(candidates)}")

cycle_df = pd.DataFrame(cycle_records)
print("\nBeta history:")
print(cycle_df[["cycle", "beta_detection", "beta_classification"]].to_string(index=False))

## 5. Beta Shift Visualisation

This is the key diagnostic plot — it shows how the algorithm adapts the annotation
budget across stages over time.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(cycle_df["cycle"], cycle_df["beta_detection"],
        "o-", color="#E91E63", linewidth=2, markersize=7, label="Stage 0: Detection")
ax.plot(cycle_df["cycle"], cycle_df["beta_classification"],
        "s-", color="#2196F3", linewidth=2, markersize=7, label="Stage 1: Classification")

ax.axhline(0.5, color="grey", linestyle=":", alpha=0.6, linewidth=1)
ax.fill_between(cycle_df["cycle"], cycle_df["beta_detection"],
                alpha=0.08, color="#E91E63")
ax.fill_between(cycle_df["cycle"], cycle_df["beta_classification"],
                alpha=0.08, color="#2196F3")

ax.set_xlabel("Labeling cycle", fontsize=12)
ax.set_ylabel("Beta weight", fontsize=12)
ax.set_title(
    "Roth & Small Algorithm 1 — Stage Weight Evolution\n"
    "Budget shifts from Detection toward Classification as Detection plateaus",
    fontsize=11,
)
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.15)
ax.set_xticks(cycle_df["cycle"])
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

os.makedirs(ROOT / "outputs" / "plots", exist_ok=True)
plt.savefig(ROOT / "outputs" / "plots" / "pipeline_al_beta_shift.png", dpi=150)
plt.show()
print("Saved: outputs/plots/pipeline_al_beta_shift.png")

## 6. Per-Stage Uncertainty Distributions

The two stages score the same images very differently — detection is a binary problem
(lower max entropy = log 2 ≈ 0.69), while classification has 6 classes (max entropy = log 6 ≈ 1.79).

In [ ]:
stage_scores = learner.compute_stage_scores(loader, device=device)

paths = list(stage_scores.keys())
scores_arr = np.array([stage_scores[p] for p in paths])   # (N, 2)

stage_labels = ["Detection (binary, max H=0.69)", "Classification (6-class, max H=1.79)"]
colors = ["#E91E63", "#2196F3"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for d in range(2):
    axes[d].hist(scores_arr[:, d], bins=40, color=colors[d],
                 edgecolor="white", linewidth=0.5)
    axes[d].axvline(scores_arr[:, d].mean(), color="black",
                    linestyle="--", linewidth=1.5, label=f"mean={scores_arr[:, d].mean():.3f}")
    axes[d].set_title(f"Stage {d}: {stage_labels[d]}", fontsize=10)
    axes[d].set_xlabel("Entropy uncertainty score")
    axes[d].set_ylabel("Image count")
    axes[d].legend()

plt.suptitle("Uncertainty Distributions Across Pipeline Stages", fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / "outputs" / "plots" / "pipeline_al_stage_uncertainty.png", dpi=150)
plt.show()

## 7. Combined Weighted Score vs. Individual Stage Scores

The combined score `Σ β_d * q_d(x)` determines which images are selected.
When β = [1, 0], it equals the detection score. As β shifts, classification uncertainty gains influence.

In [ ]:
combined = np.array([float(learner.beta @ stage_scores[p]) for p in paths])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for d, (label, color) in enumerate(zip(["Detection", "Classification"], colors)):
    axes[d].scatter(scores_arr[:, d], combined, alpha=0.25, s=10, color=color)
    # Diagonal reference line
    lim = max(scores_arr[:, d].max(), combined.max())
    axes[d].plot([0, lim], [0, lim], "k--", linewidth=0.8, alpha=0.5)
    axes[d].set_xlabel(f"Stage {d} score ({label})", fontsize=11)
    axes[d].set_ylabel("Combined weighted score", fontsize=11)
    axes[d].set_title(f"Combined vs. {label}")

beta_str = "  ".join(f"β_{stage_labels[d].split()[0]}={learner.beta[d]:.3f}"
                     for d, stage_labels in enumerate([["detection"], ["classification"]]))
plt.suptitle(f"Current beta — detection: {learner.beta[0]:.3f}  classification: {learner.beta[1]:.3f}",
             fontsize=11)
plt.tight_layout()
plt.savefig(ROOT / "outputs" / "plots" / "pipeline_al_combined_scores.png", dpi=150)
plt.show()

## 8. Top Candidates Selected in Final Cycle

In [ ]:
final_candidates = learner.select_candidates(loader, budget=30, device=device)

# Show top 12 images
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax, img_path in zip(axes.flat, final_candidates[:12]):
    from PIL import Image
    img = np.array(Image.open(img_path).convert("L"))
    cls = Path(img_path).parent.name
    score = combined[paths.index(img_path)] if img_path in paths else 0.0
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{cls}\n{score:.3f}", fontsize=8)
    ax.axis("off")

plt.suptitle("Top 12 Candidates Selected for Labeling (class | combined score)", fontsize=11)
plt.tight_layout()
plt.savefig(ROOT / "outputs" / "plots" / "pipeline_al_top_candidates.png", dpi=150)
plt.show()

print(f"\nAll {len(final_candidates)} candidates saved to: outputs/pipeline_al_candidates.csv")

## 9. Save All Outputs

In [ ]:
# Beta history CSV
cycle_df.to_csv(ROOT / "outputs" / "pipeline_al_cycle_log.csv", index=False)

# Final candidate list
pd.DataFrame({"path": final_candidates}).to_csv(
    ROOT / "outputs" / "pipeline_al_candidates.csv", index=False
)

# Beta history plot via learner built-in
learner.plot_beta_history(
    stage_labels=["Detection", "Classification"],
    output_path=str(ROOT / "outputs" / "plots" / "pipeline_al_beta_history.png"),
)

print("All outputs saved:")
print("  outputs/pipeline_al_cycle_log.csv")
print("  outputs/pipeline_al_candidates.csv")
print("  outputs/plots/pipeline_al_beta_shift.png")
print("  outputs/plots/pipeline_al_stage_uncertainty.png")
print("  outputs/plots/pipeline_al_combined_scores.png")
print("  outputs/plots/pipeline_al_top_candidates.png")
print("  outputs/plots/pipeline_al_beta_history.png")

## Summary

| Concept | What we saw |
|---------|-------------|
| **Beta initialisation** | β = [1, 0] — all weight on detection |
| **Beta shift** | As cycles progress, weight moves toward classification |
| **Entropy scales** | Detection: [0, 0.69] — Classification: [0, 1.79] |
| **Combined score** | Weighted sum β·q adapts to which stage benefits most |
| **Annotation savings** | Paper reports 35–45% fewer labels needed vs. uniform budget |

### Next steps for production

1. Replace `checkpoint=None` with real trained inspection model weights
2. After each `select_candidates()`, send images to Vision Toolbox for labeling
3. Retrain both models on expanded labeled set
4. Call `learner.step()` with the updated models
5. Set `pipeline_al.enabled: true` in `config/pipeline_config.yaml` to run via CLI

## Roth & Small (2008) Algorithm 1 — Sign Convention

The key insight is tracking **certainty** (not uncertainty) so that $\Delta U > 0$ when a stage improves:

$$U^{(d)}_t = \text{mean}_{x \in S_u}[1 - H_\text{norm}^{(d)}(x)]$$

$$\beta^{(d)}_t = \frac{\beta^{(d)}_{t-1} + \lambda \cdot \hat{\Delta}^{(d)}_t}{\|\beta_{t-1} + \lambda \cdot \hat{\Delta}_t\|_1}$$

When a stage **plateaus** ($\Delta U \to 0$), its $\beta$ stops growing and weight naturally shifts to later stages that are still improving. The U(d) plot below shows this convergence.

**Exact formulas from Algorithm 1 (Roth & Small 2008)**:
- $q(x) = 1 - H_\text{norm}(x)$ — certainty score (higher = more certain)
- $U^{(d)}_t = \text{mean}_{x \in S_u}[q^{(d)}(x)]$ — stage utility
- $\Delta_t = \Delta U_t / \|\Delta U_t\|$ — normalized change
- $\beta$ update: $\beta_t \leftarrow (\beta_{t-1} + \lambda \Delta_t) / \|\cdot\|_1$ — L1-normalized